# JEPA Pretraining for Motion History Encoder

This notebook runs the JEPA pretraining for the `MotionHistoryEncoder` using the
combined `utils_kaggle.py`.

Steps:
1. **Imports** — pull symbols from the combined module
2. **Config** — set device, paths, and hyperparameters
3. **Data** — build train/val dataloaders with `create_dataloader`
4. **Train** — build `PretrainTrainer` and call `.run()`


In [ ]:
from pathlib import Path
import torch
import utils_kaggle as U

U.WANDB_AVAILABLE
U.IGNITE_AVAILABLE

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Configuration

Edit the values below before training.  All Config defaults are in the
combined `utils_kaggle.Config` dataclass.


In [ ]:
config = U.Config()
config.device = device

# Paths — change if your dataset lives elsewhere
config.dataset_path = Path("./dataset/humanml3d-subset")
config.checkpoint_dir = Path("./checkpoints/pretrain")
config.output_path = Path("./output/pretrain")

# Training hyperparameters
config.batch_size = 200
config.effective_batch_size = 400
config._num_epochs = 200

# Model architecture (encoder)
# config.encoder_config.hidden_size = 512
# config.encoder_config.intermediate_size = 1024
# config.encoder_config.num_hidden_layers = 4
# config.encoder_config.num_attention_heads = 16
# config.encoder_config.num_registers = 2

# Curriculum — set to None to disable, or keep the default schedule
config.curriculum = None
config.horizon = 40

# W&B (optional — set to None to disable)
wandb_project = "pretrain"  # or None

print(f"Config created. Total epochs: {config.get_num_epochs()}")
print(f"Dataset path: {config.dataset_path}")
print(f"Checkpoint dir: {config.checkpoint_dir}")


## Data Loaders


In [ ]:
print("Loading train dataloader...")
train_loader, train_normalizer = U.create_dataloader(config, split="train", shuffle=True)
print(f"Train batches (current horizon={config.horizon}): {len(train_loader)}")

print("Loading validation dataloader...")
val_loader, _ = U.create_dataloader(config, split="val", shuffle=False)
print(f"Validation batches: {len(val_loader)}")

# Peek at a batch
sample = next(iter(train_loader))
print(f"\nMotion shape: {sample['motion'].shape}")  # (B, T, 271)
print(f"Text emb shape: {sample['text_clip'].shape}")  # (B, 1, 512)


## Train


In [ ]:
# Free VRAM before allocating model
import gc

gc.collect()
torch.cuda.empty_cache()

trainer = U.PretrainTrainer(
    config=config,
    train_loader=train_loader,
    val_loader=val_loader,
    normalizer=train_normalizer,
    wandb_project=wandb_project,
)

trainer.run()

import os

latest = os.path.join(config.checkpoint_dir, "pretrain_latest.pt")
print(f"Done. Latest checkpoint: {latest}")
